In [ ]:
import pandas as pd
import os
import shutil
import numpy as np
import subprocess
import glob
import sys
import gzip

In [ ]:
# Supress copy warning.
pd.options.mode.chained_assignment = None

In [ ]:
# ! gcsfuse --dir-mode 555 --file-mode 444 --implicit-dirs gp2tier2_vwb ./testing_gwas/gp2tier2 # R7
! gcsfuse --dir-mode 555 --file-mode 444 --implicit-dirs gp2_release9 ./testing_gwas/rel9_info # R9
! gcsfuse --dir-mode 777 --file-mode 777 --implicit-dirs gp2_working_eu ./testing_gwas/working

In [ ]:
wd = '/home/kuznetsovn2/run_pipeline'

In [ ]:
def shell_do(command, print_cmd=True, log=False, return_log=False, err=False):
    if print_cmd:
        print(f'Executing: {(" ").join(command.split())}', file=sys.stderr)

    res=subprocess.run(command.split(), stdout=subprocess.PIPE, stderr=subprocess.PIPE)

    output = res.stdout.decode('utf-8') + res.stderr.decode('utf-8')

    if log:
        print(output)
    if return_log:
        return output
    if err:
        return res.stderr.decode('utf-8')

In [ ]:
def get_vcf_names(vcf_path):
    with gzip.open(vcf_path, "rt") as ifile:
          for line in ifile:
            if line.startswith("#CHROM"):
                  vcf_names = [x for x in line.split('\t')]
                  break
    ifile.close()
    return vcf_names

### Explore Samples - HY Data/Master Key

In [ ]:
# Explore new data
analysis = 'CS' # CS, LT, SURV (run CS or LT before SURV)
month = 'JAN' # Save month
new_data = pd.read_csv(f'{wd}/testing_gwas/working/lietsel/Data/HY/hy_{analysis.lower()}_2025-03-12.csv', low_memory = False)
# new_data.drop(columns = 'Unnamed: 0', inplace = True) # currently only on SURV data

new_data.drop_duplicates(inplace = True)
display(new_data)
display(new_data.clinical_study.value_counts())
print(len(np.unique(new_data.GP2ID)))

In [ ]:
# Get postural instability data for available cohorts - must be CS (discrete)

if analysis != 'LT':
    postural = pd.read_csv(f'{wd}/testing_gwas/working/lietsel/Data/HY/ps_{analysis.lower()}_2025-03-12.csv')
    # postural.drop(columns = 'Unnamed: 0', inplace = True)

    new_data = pd.concat([new_data, postural])
    display(new_data)

    if analysis == 'CS':
        new_data['outcome'] = ((new_data['hoehn_and_yahr_stage'] >= 3) | (new_data['mds_updrs_postural_stability'] >= 3) |
                                (new_data['updrs_postural_stability'] >= 3)).astype(int)
        display(new_data.outcome.value_counts())

In [ ]:
# Master Key - Release 9 Samples
# master = pd.read_csv(f'{wd}/testing_gwas/gp2tier2/release9_18122024/clinical_data/master_key_release9_final_vwb.csv')
master = pd.read_csv(f'{wd}/testing_gwas/rel9_info/clinical/master_key_release9_final_vwb.csv')

master_samples = master[master.GP2ID.isin(new_data.GP2ID)]
display(master_samples.nba_label.value_counts()) # change to "label" if prior to release 8

display(master_samples.study.value_counts())
print(len(master_samples))

In [ ]:
# Missing samples - make sure have most recent master key
missing = new_data[~new_data.GP2ID.isin(master.GP2ID)]
missing = missing[~missing.clinical_study.isin(['PROBAND', 'NTUH'])]

display(missing)
print(len(np.unique(missing.GP2ID)))
# missing.to_csv(f'{wd}/new_data_{analysis}_missing_samples.csv', index = False)

In [ ]:
# ! gsutil cp {wd}/new_data_{analysis}_missing_samples.csv gs://gp2_working_eu/nicole/{month}/new_data_{analysis}_missing_samples.csv

### Explore Samples

In [ ]:
if analysis != 'SURV':
    full_data = new_data.merge(master[['GP2ID', 'study_type', 'nba_label']], on = ['GP2ID', 'study_type'], how = 'inner')
else:
    full_data = new_data.merge(master[['GP2ID', 'study_type', 'biological_sex_for_qc', 'nba_label']], on = ['GP2ID', 'study_type'], how = 'inner')

display(full_data)

In [ ]:
# Split by ancestry
anc = 'EUR'
subset = full_data[full_data.nba_label == anc]
display(subset)

# Check that all have the same GP2_phenotype and study_type
display(subset.GP2_phenotype_for_qc.value_counts())
display(subset.study_type.value_counts())

if analysis != 'SURV':
    # Check that all samples have at least age_of_onset or age_at_diagnosis
    age = subset.groupby(['GP2ID', 'age_at_baseline', 'age_of_onset', 'age_at_diagnosis'], dropna=False, as_index=False).size()
    no_baseline_age = age[age.age_at_baseline.isnull()]
    no_outcome_age = age[(age.age_of_onset.isnull()) & (age.age_at_diagnosis.isnull())]

    display(no_baseline_age)
    display(no_outcome_age)

    # Remove samples if necessary
    if len(no_baseline_age) > 0:
        subset = subset[~subset.GP2ID.isin(no_baseline_age.GP2ID)]
    if len(no_outcome_age) > 0:
        subset = subset[~subset.GP2ID.isin(no_outcome_age.GP2ID)]

    # Check that all samples have HY scores
    HY = subset.groupby(['GP2ID', 'hoehn_and_yahr_stage'], dropna=False, as_index=False).size()
    display(HY[HY.hoehn_and_yahr_stage.isnull()])
else:
    # Check that all samples have HY scores
    HY = subset.groupby(['GP2ID', 'event'], dropna=False, as_index=False).size()
    display(HY[HY.event.isnull()])

if 'outcome' in subset.columns:
    HY = subset.groupby(['GP2ID', 'outcome'], dropna=False, as_index=False).size()
    display(HY[HY.outcome.isnull()])

In [ ]:
# Subset to just Case(/Control)
subset = subset[subset.study_type == 'Case(/Control)']

In [ ]:
# Create age_outcome column to combine age_of_onset and age_at_diagnosis
subset['age_outcome'] = subset['age_at_diagnosis'].combine_first(subset['age_of_onset']) # order is because diagnosis > onset for age
display(subset[subset.age_outcome.isnull()])

# Create new_disease_duration column
subset['disease_duration_new'] = subset['age_at_baseline'] - subset['age_outcome'] # all samples are PD at sample collection
display(subset[subset.disease_duration_new.isnull()])

In [ ]:
# Check differences
age_disease_cols = ['GP2ID', 'age_at_sample_collection','age_of_onset','age_at_diagnosis','disease_duration','age_outcome','disease_duration_new']
# display(subset[subset.disease_duration_new != subset.disease_duration])

max_diff = max(subset.disease_duration_new - subset.disease_duration)
print('Max disease duration difference:', max_diff)

In [ ]:
### Additional but not required fields
optional_cols = ['ledd_daily', 'clinical_state_on_medication', 'medication_for_pd', 'dbs_status']

for optional_col in optional_cols:
    # Check how many samples have values for each optional col
    optional = subset.groupby(['GP2ID', optional_col], dropna=False, as_index=False).size()
    optional_subset = optional[optional[optional_col].notnull()]
    display(optional_subset[optional_col].value_counts())
    print(f'{optional_col} unique sample count:', len(optional_subset.GP2ID.unique()))

# Non-Genotyping Inputs

### Subset Per Analysis for One Ancestry

In [ ]:
# BCFTools Approach
# subset['subset_ID'] = '0_' + subset.GP2sampleID # VCF uses GP2 sample IDs with 0_ in front
# ancestry_samples = ','.join(subset.subset_ID.values) # convert to string for command line

# Release 7 and 8
# save_samples = subset[['GP2ID', 'GP2sampleID', 'subset_ID']]
# save_samples.to_csv(f'{out_path}/get_{anc.upper()}_samples.csv', index = False)

# Release 7 configuration
# keep = save_samples[[ '#FID', 'GP2sampleID']]
# keep.rename(columns = {'GP2sampleID': 'IID'}, inplace = True)

# Old saved samples
# save_samples = pd.read_csv(f'{wd}/testing_gwas/working/nicole/DEC/{anc.upper()}/{analysis}/get_{anc.upper()}_samples.csv')

In [ ]:
# Save samples to pull imputed genotypes later - PLINK subsetting
print(len(np.unique(subset.GP2ID)))

# analysis = 'CS_disc' # comment out if needed
out_path = f'{wd}/{month}_run_trial/{anc.upper()}/{analysis}'
os.makedirs(out_path, exist_ok=True)

# Release 9
subset['#FID'] = 0
save_samples = subset[['#FID', 'GP2ID']]
save_samples.drop_duplicates(inplace = True)
save_samples.rename(columns = {'GP2ID': 'IID'}, inplace = True)
display(save_samples)

save_samples.to_csv(f'{out_path}/{anc.upper()}_keep_samples.txt', sep = '\t', index = False)

### Format Inputs
 - Covariates requires Plink-style columns #FID and PHENO - can be populated with 0; necessary for cross-sectional and longitudinal
 - Refer to example files in cloned GWAS Github

In [ ]:
# Covariate formatting
# subset['#FID'] = 0 # done in PLINK keep sample file

covar = subset[['#FID', 'GP2ID', 'biological_sex_for_qc', 'GP2_phenotype_for_qc', 'age_at_baseline',
                'disease_duration_new', 'study_type', 'clinical_state_on_medication', 'clinical_study']]
covar.rename(columns = {'GP2ID': 'IID', 'biological_sex_for_qc': 'SEX', 'GP2_phenotype_for_qc': 'PHENO'}, inplace = True)

# change study_arm variables going forward?
covar.PHENO = covar.PHENO.replace({'PD': 1})
covar.SEX = covar.SEX.replace({'Male': 1, 'Female': 2})

# make sure to correct for any spaces --> underscores/dashes; fine if they're in IDs
covar.replace({'Case(/Control)': 'case_control', 'PROMO-EPIPARK': 'PROMO_EPIPARK', 'PPMI-EUR': 'PPMI_EUR',
                'PROMO-GENEPARK': 'PROMO_GENEPARK', 'PD (Case(/Control))': 'PD'}, inplace = True)

# fill NA in clinical state on medication w/ Unknown
covar.clinical_state_on_medication.fillna('Unknown', inplace = True)

if analysis == 'LT':
    covar = pd.get_dummies(covar, columns=['clinical_study'], prefix='study')

    # make sure boolean columns are 0s and 1s
    bool_columns = covar.select_dtypes(include=['bool']).columns
    covar[bool_columns] = covar[bool_columns].astype(int)
    covar.drop(columns = 'clinical_state_on_medication', inplace = True)

covar.drop_duplicates(inplace = True)
display(covar)

covar.to_csv(f'{out_path}/hy_{analysis}_{anc}_covariates.tsv', sep = '\t', index = False)

In [ ]:
# Check clinical_state_on_medication by cohort
if analysis != 'LT':
    display(covar.groupby(['clinical_study', 'clinical_state_on_medication']).size())

    covar.drop(columns = 'clinical_state_on_medication', inplace = True)
    covar.drop_duplicates(inplace = True)
    print(len(covar))
    covar.to_csv(f'{out_path}/hy_{analysis}_{anc}_covariates.tsv', sep = '\t', index = False)

In [ ]:
# Outcome - Continuous

# subset_ID: Oct AJ; GP2sampleID: Nov EUR
if analysis == 'CS':
    outcome = subset[['GP2ID', 'hoehn_and_yahr_stage']]
    outcome['study_days'] = 0 # CS
    outcome.rename(columns = {'GP2ID': 'IID', 'hoehn_and_yahr_stage': 'y'}, inplace = True)
if analysis == 'CS_disc':
    outcome = subset[['GP2ID', 'outcome']]
    outcome['study_days'] = 0 # CS
    outcome.rename(columns = {'GP2ID': 'IID', 'outcome': 'y'}, inplace = True)
    outcome.y.replace({0:1, 1:2}, inplace = True)
elif analysis == 'SURV':
    outcome = subset[['GP2ID','visit_month_first', 'censored_month', 'event']]
    outcome['study_days'] = outcome.censored_month - outcome.visit_month_first
    outcome.rename(columns = {'GP2ID': 'IID', 'event': 'y', 'visit_month_first': 'tstart', 'censored_month': 'tend'}, inplace = True)
elif analysis == 'LT':
    outcome = subset[['GP2ID', 'hoehn_and_yahr_stage', 'visit_month']]
    outcome.rename(columns={'visit_month': 'study_days'}, inplace = True)
    outcome.rename(columns = {'GP2ID': 'IID', 'hoehn_and_yahr_stage': 'y'}, inplace = True)

# if want Binary outcome
# outcome['y'] = 0 # initialize column
# outcome['y'] = np.where(outcome['hoehn_and_yahr_stage'] >= 3, 1, outcome['y']) # eventually change threshold

outcome.drop_duplicates(inplace = True)
outcome.dropna(inplace = True)
display(outcome)

# check how many repeats
repeats = outcome.groupby(['IID'], dropna=False, as_index=False).size()
display(repeats['size'].value_counts())
print(np.mean(repeats['size']))

outcome.to_csv(f'{out_path}/hy_{analysis}_{anc}_phenotypes.tsv', sep = '\t', index = False)

In [ ]:
# Copy to working directory in buckets
! gsutil cp -r {out_path} gs://gp2_working_eu/nicole/{month}/{anc}/

### PLINK Keep Samples

In [ ]:
# Create shell script to execute locally - Release 9

# Establish paths for inputs/outputs
# analysis = 'SURV' # CS, LT, SURV, CS_disc
label = 'EUR'
month = 'JAN'

# Release 9
release_path = f'{wd}/testing_gwas/rel9_info/imputation_out_dir/{label}/final_files/chr${{CHROM}}_{label}_release9_vwb'

intermed_dir = f'/dev/shm/hold_genotypes/chr${{CHROM}}_hy_{analysis}_subsetted_{label}'
out_dir = f'gs://gp2_working_eu/nicole/{month}/{label}/{analysis}'

# Import sample IDs needed for subset
keep_samples_file = f'{wd}/testing_gwas/working/nicole/{month}/{label}/{analysis}/{label}_keep_samples.txt'

# Create shell script with command
subset_cmd = f"plink2  --pfile {release_path} --keep {keep_samples_file} --export vcf 'bgz' 'vcf-dosage=HDS' --out {intermed_dir}"
cp_cmd = f"gsutil cp {intermed_dir}.vcf.gz {out_dir}"
rm_cmd = f"rm {intermed_dir}.vcf.gz"

os.makedirs(f'vcf_scripts/{label}', exist_ok=True)

# This shell script will accept CHROM as an input when you launch it
with open(f'vcf_scripts/{label}/{analysis}_analysis_subset_plink.sh', 'w') as f:
    f.write('#!/bin/bash\n\n')
    f.write('CHROM=$1\n\n')
    f.write(f'{subset_cmd}\n')
    f.write(f'{cp_cmd}\n')
    f.write(f'{rm_cmd}\n')
    f.close()


print(f"Preparing bash script for subsetting {label} ancestry!")

In [ ]:
# Launch in separate shell windows
print(f'chmod +x {wd}/vcf_scripts/{label}/{analysis}_analysis_subset_plink.sh')
print(f'{wd}/vcf_scripts/{label}/{analysis}_analysis_subset_plink.sh 22')

# Run once for log dir
! mkdir /dev/shm/hold_genotypes